# Notebook 04: Cost Optimization Pattern

**CCA Pattern:** Prompt Caching vs Batch API misuse

This notebook demonstrates the correct cost optimization strategy for live customer support. We show:

1. **Anti-Pattern**: Why the Batch API is ALWAYS wrong for live support (conceptual)
2. **Correct Pattern**: Prompt Caching with `cache_control` for 90% savings on repeated context
3. **Compare**: Token accounting difference between cached and uncached runs

## Setup

Install dependencies and set up services. Requires `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.agent.agent_loop import run_agent_loop
from customer_service.agent.system_prompts import (
    POLICY_DOCUMENT,
    get_system_prompt,
    get_system_prompt_with_caching,
)
from customer_service.data.customers import CUSTOMERS
from customer_service.data.scenarios import SCENARIOS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


from dotenv import find_dotenv, load_dotenv

# Load ANTHROPIC_API_KEY from .env (find_dotenv walks up from the notebooks/ dir)
load_dotenv(find_dotenv(), override=False)

client = anthropic.Anthropic()
scenario = SCENARIOS["happy_path"]  # C001, $50 refund — the ID must be in the message
customer_message = f"Customer ID: {scenario['customer_id']}. {scenario['message']}"
print(f"Scenario: {customer_message}")

## Anti-Pattern: Batch API for Live Customer Support

<div style="background-color:#fff0f0; border-left:4px solid #cc0000; padding:16px; margin:12px 0; border-radius:4px;">

**WRONG APPROACH: Using Batch API to cut costs**

A common misconception is that using the Batch API reduces costs by 50% for customer support operations. This is **always wrong** for live support.

</div>

### Why Batch API Is Wrong for Live Support

**1. Latency** — The Batch API processes requests asynchronously with up to **24-hour** turnaround time. A customer waiting for a refund resolution cannot wait 24 hours.

**2. Not ZDR-eligible** — The Batch API is not eligible for Zero Data Retention (ZDR). Organizations with strict data security requirements often require ZDR for customer data. The Real-Time API supports ZDR; the Batch API does not.

**3. Wrong cost lever** — The Batch API saves costs by accepting delay. For live support, the correct cost lever is **Prompt Caching**: sending the same large static context on every API call provides up to **90% cost savings** on repeated tokens — far better than the Batch API's 50% discount — with no latency penalty.

> **CCA Exam Tip:** Use this decision rule:
> 
> - **Someone waiting?** → Real-Time API + Prompt Caching
> - **No one waiting + no ZDR requirement?** → Batch API acceptable
> 
> **"Batch API for live support"** = ALWAYS WRONG. Eliminate immediately on the exam.

The module `customer_service.anti_patterns.batch_api_live` contains the full teaching explanation. There is no runnable code for the Batch API anti-pattern because it should never be executed for live support.

## Correct Pattern: Prompt Caching with `cache_control`

<div style="background-color:#f0fff0; border-left:4px solid #00aa00; padding:16px; margin:12px 0; border-radius:4px;">

**CORRECT APPROACH: Real-Time API + Prompt Caching**

Mark large static context blocks (policy documents, tool descriptions) with `cache_control: {"type": "ephemeral"}`. After the first call writes to cache, subsequent calls read from cache at **10% of the normal input token cost**.

</div>

We will compare **three runs** of the same scenario. All three send the **same text**: the agent instructions plus the full policy document. The only variable is whether the policy document carries a cache marker.

- **Run 1**: Uncached — one plain string, no `cache_control`. The policy document is billed at the full input rate on every call.
- **Run 2**: Cached, first run — the first API call writes the policy block to cache. The agent loop makes several calls per run (one per tool round-trip), so the later calls in this same run already read from cache.
- **Run 3**: Cached, second run — every call reads from cache, nothing is written.

Look for `cache_creation_input_tokens` (Run 2) and `cache_read_input_tokens` (Runs 2 and 3) in the output. The ephemeral cache lives for 5 minutes and each read refreshes it, so run these cells one after another.

In [ ]:
print(f"POLICY_DOCUMENT token estimate: {len(POLICY_DOCUMENT) // 4}")
print("  (Minimum for caching on claude-sonnet-4-6: 2048 tokens)")
print()

cached_prompt = get_system_prompt_with_caching()
print(f"Cached format: list of {len(cached_prompt)} blocks")
print(f"  Block 0 (instructions) has cache_control: {'cache_control' in cached_prompt[0]}")
print(f"  Block 1 (POLICY_DOCUMENT) has cache_control: {'cache_control' in cached_prompt[1]}")

# The uncached baseline must send the SAME text, just without the cache marker.
# get_system_prompt() alone is NOT a fair baseline: it omits the policy document
# entirely, so it would look cheaper than caching simply by sending less context.
uncached_prompt = get_system_prompt() + "\n\n" + POLICY_DOCUMENT
print()
print(f"Uncached format: plain string, {len(uncached_prompt) // 4} tokens (estimate)")
print(f"  Same text as the cached blocks: {uncached_prompt == cached_prompt[0]['text'] + chr(10) * 2 + cached_prompt[1]['text']}")
print(f"  get_system_prompt() alone: {len(get_system_prompt()) // 4} tokens — no policy document")

### Run 1: Uncached (baseline)

One plain string holding the agent instructions **and** the full policy document, with no `cache_control`. This is the same text the cached runs send, so the only difference between Run 1 and Runs 2–3 is caching. Every token of the policy document is charged at the full input rate on every API call in the loop.

In [ ]:
class _UsageWrapper:
    pass


services_uncached = make_services()
result_uncached = run_agent_loop(
    client,
    services_uncached,
    customer_message,
    uncached_prompt,  # plain string: instructions + POLICY_DOCUMENT, no caching
)
w = _UsageWrapper()
w.usage = result_uncached.usage
print_usage(w)
print(f"\nCache read tokens:  {result_uncached.usage.cache_read_input_tokens}")
print(f"Cache write tokens: {result_uncached.usage.cache_creation_input_tokens}")

### Run 2: Cached — first run (cache write, then reads)

Same text, now as two blocks with `cache_control` on the POLICY_DOCUMENT block. The first API call of this run writes the policy block to cache, so you will see `cache_creation_input_tokens > 0`. That write is charged at **1.25x** the normal input rate.

The usage below is the **total for the whole run**. The agent loop makes one API call per tool round-trip (lookup, policy check, refund, log), and every call after the first already finds the block in cache. So this run also shows `cache_read_input_tokens > 0`: one write, several reads.

In [ ]:
services_cached_1 = make_services()
result_cached_1 = run_agent_loop(
    client,
    services_cached_1,
    customer_message,
    cached_prompt,  # list-of-blocks with cache_control on the policy block
)
w = _UsageWrapper()
w.usage = result_cached_1.usage
print_usage(w)
print(f"\nCache write tokens: {result_cached_1.usage.cache_creation_input_tokens}")
print(f"Cache read tokens:  {result_cached_1.usage.cache_read_input_tokens}")

### Run 3: Cached — second run (cache read only)

Same request again, immediately after Run 2. The policy block is still in cache (the 5-minute window is refreshed on every read), so every call in this run reads it: `cache_read_input_tokens > 0` and `cache_creation_input_tokens == 0`. Cached tokens are charged at **10% of the normal input rate** — the 90% saving.

If more than 5 minutes have passed since Run 2, the cache has expired and this run will show a write again. Re-run the cell.

In [ ]:
services_cached_2 = make_services()
result_cached_2 = run_agent_loop(
    client,
    services_cached_2,
    customer_message,
    cached_prompt,
)
w = _UsageWrapper()
w.usage = result_cached_2.usage
print_usage(w)
print(f"\nCache write tokens: {result_cached_2.usage.cache_creation_input_tokens}")
print(f"Cache read tokens:  {result_cached_2.usage.cache_read_input_tokens}")

## Compare Results

Uncached (Run 1) vs the cached second run (Run 3). Read the table carefully:

- `input_tokens` **drops** in the cached run only because the policy tokens moved out of that bucket and into `cache_read`. It is not a saving by itself.
- `total_input` (input + cache read + cache write) shows both runs sent the **same amount of context**. Small differences come from Claude's tool calls and replies varying from run to run.
- `estimated_cost_usd` is the number the pattern is about: the same context, priced with cached tokens at 10%.

The delta on `estimated_cost_usd` will not be a full 90%, because the tools, the customer message and Claude's own output are never cached and are priced normally. The larger the static context relative to the rest, the closer to 90% the total saving gets.

In [ ]:
from helpers import estimate_cost


def usage_row(result) -> dict:
    u = result.usage
    return {
        "api_calls": sum(1 for m in result.messages if m["role"] == "assistant"),
        "input_tokens": u.input_tokens,
        "cache_write": u.cache_creation_input_tokens,
        "cache_read": u.cache_read_input_tokens,
        "total_input": u.input_tokens + u.cache_read_input_tokens + u.cache_creation_input_tokens,
        "output_tokens": u.output_tokens,
        "estimated_cost_usd": round(estimate_cost(u), 6),
    }


compare_results(usage_row(result_uncached), usage_row(result_cached_2))

## CCA Exam Tip: Prompt Caching Economics

> **CCA Exam Tip:**
> 
> **Prompt Caching** is the correct cost optimization for live customer support:
> 
> - **Cache write** (first call): 1.25x input cost — slight overhead to populate cache
> - **Cache read** (subsequent calls): 0.10x input cost — **90% savings** on cached tokens
> - Break-even: After just 2 calls, caching is cheaper than no caching
> - Works with: Large static policy documents, tool schemas, knowledge bases
> 
> **Key rule:** Mark the LAST static block with `cache_control`. Blocks before the cache > breakpoint are all cached together. Don't put `cache_control` on small dynamic blocks.
> 
> **Batch API for live support** → ALWAYS WRONG. It has up to 24-hour latency and is > not ZDR-eligible. Real-Time API + Prompt Caching is the answer.

## Summary

| Pattern | Approach | Latency | Cost | ZDR |
|---------|----------|---------|------|-----|
| Anti-Pattern | Batch API | up to 24 hours | 50% discount | NOT eligible |
| Correct | Real-Time + Prompt Caching | sub-second | 90% savings on cached tokens | Eligible |

**Key files:**

- `src/customer_service/agent/system_prompts.py` — `get_system_prompt_with_caching()`, `POLICY_DOCUMENT`
- `src/customer_service/anti_patterns/batch_api_live.py` — `BATCH_API_EXPLANATION` teaching constant
- `notebooks/helpers.py` — `print_usage()` with cache token display, `estimate_cost()` for the comparison

**CCA Rule:** 'Someone waiting? → Real-Time API + Prompt Caching. Batch API is ALWAYS wrong for live support.'